# Chapter 2 — Tokenization

Computers don't understand letters. They only understand numbers.
So before we can train a language model we need to turn every word
and every punctuation mark into a number. This process is called
tokenization and it is the first step in every modern LLM.

In this notebook we will use the same tokenizer that GPT-2 uses.
It breaks words into small pieces called subwords so even rare
words and emojis can be handled gracefully. We will see how
"unbelievably" becomes three tokens and how emojis don't break
anything. By the end you will understand how text becomes the
raw input for a transformer model.

## Imports

In [1]:
from dataclasses import dataclass
import tiktoken

## Configuration

In [2]:
@dataclass
class TokenizerConfig:
    name: str = "gpt2"
    vocab_size: int = 50257

## Tokenizer

In [3]:
class SimpleTokenizer:
    def __init__(self, config=None):
        self.config = config or TokenizerConfig()
        self.enc = tiktoken.get_encoding(self.config.name)
        self.eos_token = "<|endoftext|>"
        self.eos_token_id = self.enc.encode(
            self.eos_token, allowed_special={self.eos_token}
        )[0]

    def encode(self, text):
        return self.enc.encode(text, allowed_special={self.eos_token})

    def decode(self, ids):
        return self.enc.decode(ids)

    @property
    def vocab_size(self):
        return self.config.vocab_size

## Try it out

In [4]:
tokenizer = SimpleTokenizer()
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"EOS token ID: {tokenizer.eos_token_id}")
print()

text = "The cat sat on the mat."
encoded = tokenizer.encode(text)
decoded = tokenizer.decode(encoded)

print(f"Original: '{text}'")
print(f"Encoded:  {encoded}")
print(f"Decoded:  '{decoded}'")
print(f"Roundtrip: {text == decoded}")

Vocabulary size: 50,257
EOS token ID: 50256

Original: 'The cat sat on the mat.'
Encoded:  [464, 3797, 3332, 319, 262, 2603, 13]
Decoded:  'The cat sat on the mat.'
Roundtrip: True


## Rare words and emojis

In [5]:
rare = tokenizer.encode("antidisestablishmentarianism")
pieces = [tokenizer.decode([t]) for t in rare]
print(f"Pieces: {pieces}")
print(f"Decoded: '{tokenizer.decode(rare)}'")
print()

text = tokenizer.encode("Hello world!")
print(f"Decoded: {tokenizer.decode(text)}")
print()
emoji = tokenizer.encode("Hello \U0001f60a world")
print(f"Emoji test: {tokenizer.decode(emoji)}")

Pieces: ['ant', 'idis', 'establishment', 'arian', 'ism']
Decoded: 'antidisestablishmentarianism'

Decoded: Hello world!

Emoji test: Hello 😊 world
